### Visualizações dos dados do Benchmark Dataset (Tsinghua Lab) para o participante S1

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = [16., 5.]

# Objetos Epochs salvos no pré-processamento
epochs_unfiltered = mne.read_epochs('data/output/S1_epo.fif', preload=True, verbose=False)
epochs_filtered = mne.read_epochs('data/output/S1_filtered_epo.fif', preload=True, verbose=False)

**Espectro médio (com e sem filtro)**

Para conferir o efeito do filtro passa-faixa, realizado no pré-processamento, foi calculado a *Power Spectral Density* (PSD), que mostra quanta energia do sinal existe em cada frequência. O cálculo foi feito em cada epoch através do método *multitaper* do MNE, de 1 a 60Hz (apenas para incluir até 50Hz no gráfico), e depois foi retirada a média de cada epoch e de todos os canais; resultando em uma única curva para cada objeto.

Como o MNE retorna a potência em V²/Hz, o valor foi multiplicado por 10¹² para passar para µV²/Hz, que é a unidade de sinal que o dataset utiliza. Aleḿ disso, visto que, em escala linear, só as frequências baixas ficam evidentes, a potência foi convertida para decibéis.

In [ ]:
def mean_psd_db(epochs, fmin=1, fmax=60):
    psd = epochs.compute_psd(method='multitaper', fmin=fmin, fmax=fmax, verbose=False)
    mean_psd = psd.get_data().mean(axis=(0, 1))
    return psd.freqs, 10 * np.log10(mean_psd * 1e12)

freq_axis, psd_unfiltered = mean_psd_db(epochs_unfiltered)
_, psd_filtered = mean_psd_db(epochs_filtered)

plt.figure(figsize=(10, 5))
plt.plot(freq_axis, psd_unfiltered, label='Sem filtro')
plt.plot(freq_axis, psd_filtered, label='Com filtro (6 a 35 Hz)')
plt.axvline(x=6, linestyle='--', color='gray')
plt.axvline(x=35, linestyle='--', color='gray')
plt.axvline(x=50, linestyle='--', color='red')
plt.title('PSD médio de todas as epochs e canais')
plt.xlabel('Frequência (Hz)')
plt.ylabel('PSD (µV²/Hz, dB)')
plt.legend()
plt.show()

No gráfico, foram marcadas as bordas da faixa do filtro, em 6 e 35Hz, e a frequência de 50Hz, para observar se o sinal sem filtro ainda tem algum efeito da rede elétrica do laboratório.

**Sinal de uma epoch nos três domínios**

